# AutoBot — Reasoner Model Training (Qwen2.5-7B-Instruct + QLoRA SFT)

**Model**: Qwen2.5-7B-Instruct with 4-bit QLoRA adapter for causal LM / SFT

**Task**: Generate 2-3 sentence analyst narrative explaining bottleneck risk score

**Hardware**: A100 80GB GPU (Colab)

**Training specs** (from AutoBot Ref Doc Section 6.2):
- QLoRA: 4-bit NF4 quantization + LoRA r=32, alpha=64
- Target modules: q_proj, v_proj, k_proj, o_proj, gate_proj, up_proj
- Epochs: 3, Batch: 4 x 2 grad accum (effective 8), LR: 1e-4, Max seq: 2048 tokens
- Optimizer: paged_adamw_8bit
- Input: issue snapshot + risk score + PR review data + timeline cross-refs
- Output: GPT-4o generated 2-3 sentence narrative (teacher distillation)
- Only medium + high band records (~2,578 total, ~2,044 train)

## 1. Install Dependencies

In [ ]:
!pip install -q transformers==4.46.3 peft==0.13.2 datasets accelerate bitsandbytes trl
!pip install -q scikit-learn matplotlib seaborn rouge-score nltk

## 2. Upload Data

Upload `train.jsonl`, `val.jsonl`, `test.jsonl` from `labelling/data/labeled/reasoner/`.

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/autobot/reasoner'  # <-- adjust to your path

# Option B: Direct upload (uncomment below, comment out above)
# from google.colab import files
# uploaded = files.upload()
# DATA_DIR = '/content'

print(f'Data dir: {DATA_DIR}')
print(os.listdir(DATA_DIR))

## 3. Load & Prepare Dataset

In [ ]:
import json
import re
import random
from collections import Counter

random.seed(42)

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_raw = load_jsonl(os.path.join(DATA_DIR, 'train.jsonl'))
val_raw = load_jsonl(os.path.join(DATA_DIR, 'val.jsonl'))
test_raw = load_jsonl(os.path.join(DATA_DIR, 'test.jsonl'))

print(f'Train: {len(train_raw)}, Val: {len(val_raw)}, Test: {len(test_raw)}')
print(f'Train band dist: {Counter(r.get("scorer_band", "?") for r in train_raw)}')

## 3b. System Prompt & SFT Formatting

In [ ]:
# ── Qwen Reasoner System Prompt (Section 5A.5) ──────────────────────────
# Must match exactly at inference time.
SYSTEM_PROMPT = (
    'You are a bottleneck analyst for GitHub issues. '
    'Given an issue snapshot and its risk score, write a 2-3 sentence '
    'explanation for a non-technical scrum master. '
    'Reference specific signals. No bullet points.'
)


def apply_days_open_jitter(text: str, jitter_range: int = 3) -> str:
    """Apply +/-3 day jitter to DAYS_OPEN in the input string.

    Prevents the model from learning a step function on exact T+7/T+15/etc.
    Label stays unchanged -- only the input number varies.
    """
    match = re.search(r'DAYS_OPEN: (\d+)', text)
    if match:
        original = int(match.group(1))
        jittered = max(1, original + random.randint(-jitter_range, jitter_range))
        text = text.replace(f'DAYS_OPEN: {original}', f'DAYS_OPEN: {jittered}')
    return text


def strip_snapshot_tier(text: str) -> str:
    """Remove SNAPSHOT_TIER from input -- not available at inference."""
    return re.sub(r'\| SNAPSHOT_TIER: T\+\d+\s*', '', text)


def prepare_sft_examples(records, tokenizer, apply_jitter=False):
    """Convert reasoner JSONL records to chat-formatted SFT pairs.

    Each record has:
      - 'input': enriched issue text with risk score, PR reviews, timeline
      - 'label': {'narrative': '...'} -- the GPT-4o generated narrative

    Returns list of dicts with 'text' (full chat template) for SFT.
    """
    examples = []
    for r in records:
        user_text = r['input']
        narrative = r['label']['narrative']

        # Strip snapshot tier (not available at inference)
        user_text = strip_snapshot_tier(user_text)

        if apply_jitter:
            user_text = apply_days_open_jitter(user_text)

        # Build chat messages: system + user + assistant (target)
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_text},
            {'role': 'assistant', 'content': narrative},
        ]
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )

        examples.append({
            'text': formatted,
            'band': r.get('scorer_band', 'medium'),
        })
    return examples


print(f'System prompt ({len(SYSTEM_PROMPT)} chars):\n{SYSTEM_PROMPT}')

## 4. Load Tokenizer & Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN = 2048

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Qwen uses eos_token as pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization (QLoRA) to fit 7B model on A100
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded: {MODEL_ID} (4-bit QLoRA)")
print(f"Parameters: {model.num_parameters():,}")

## 4b. Prepare Data with Chat Template

In [ ]:
train_data = prepare_sft_examples(train_raw, tokenizer, apply_jitter=True)
val_data = prepare_sft_examples(val_raw, tokenizer, apply_jitter=False)
test_data = prepare_sft_examples(test_raw, tokenizer, apply_jitter=False)

print(f'Sample formatted text (first 400 chars):\n{train_data[0]["text"][:400]}')
print(f'\n... (last 200 chars):\n{train_data[0]["text"][-200:]}')

# Check token lengths
sample_lens = [len(tokenizer.encode(d['text'], truncation=False)) for d in train_data[:100]]
print(f'\nToken length stats (first 100): min={min(sample_lens)}, max={max(sample_lens)}, '
      f'mean={sum(sample_lens)/len(sample_lens):.0f}')
print(f'Exceeding {MAX_SEQ_LEN}: {sum(1 for l in sample_lens if l > MAX_SEQ_LEN)}/100')

## 5. Apply LoRA (Section 6.2)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj'],
    bias='none',
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

## 6. Tokenize Dataset for SFT

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling


def tokenize_sft(examples):
    """Tokenize with labels = input_ids (causal LM).

    For SFT, we mask the prompt tokens so loss is only computed on the
    assistant's response (the narrative). We find the assistant header
    token position and set labels to -100 before it.
    """
    tokens = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
    )

    all_labels = []
    # Find the assistant response start to mask prompt tokens
    assistant_header = tokenizer.encode(
        '<|im_start|>assistant\n', add_special_tokens=False
    )
    header_len = len(assistant_header)

    for input_ids in tokens['input_ids']:
        labels = list(input_ids)  # copy
        # Find assistant header position
        found = False
        for i in range(len(input_ids) - header_len + 1):
            if input_ids[i:i+header_len] == assistant_header:
                # Mask everything before the assistant response
                for j in range(i + header_len):
                    labels[j] = -100
                found = True
                break
        if not found:
            # Fallback: mask first 80% as prompt (conservative)
            prompt_end = int(len(input_ids) * 0.8)
            for j in range(prompt_end):
                labels[j] = -100
        # Also mask padding tokens
        for j in range(len(labels)):
            if input_ids[j] == tokenizer.pad_token_id:
                labels[j] = -100
        all_labels.append(labels)

    tokens['labels'] = all_labels
    return tokens


train_ds = Dataset.from_list(train_data).map(tokenize_sft, batched=True, remove_columns=['text', 'band'])
val_ds = Dataset.from_list(val_data).map(tokenize_sft, batched=True, remove_columns=['text', 'band'])
test_ds = Dataset.from_list(test_data).map(tokenize_sft, batched=True, remove_columns=['text', 'band'])

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')

# Verify label masking on first example
sample_labels = train_ds[0]['labels']
n_masked = (sample_labels == -100).sum().item()
n_total = len(sample_labels)
n_active = n_total - n_masked
print(f'First example: {n_masked} masked tokens, {n_active} active (narrative) tokens out of {n_total}')

## 7. Training Setup

In [ ]:
import numpy as np
from transformers import Trainer, TrainingArguments

OUTPUT_DIR = '/content/reasoner_checkpoints'

# Drive path for persistent storage
DRIVE_DIR = '/content/drive/MyDrive/autobot/reasoner'
DRIVE_PLOTS = f'{DRIVE_DIR}/plots'
os.makedirs(DRIVE_PLOTS, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,                    # Section 6.2: generation overfits faster
    per_device_train_batch_size=4,         # Reduced from 8 to avoid OOM
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,         # Effective batch = 4*2 = 8 (matches spec)
    learning_rate=1e-4,                    # Section 6.2: lower LR for generation stability
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=25,
    report_to='none',
    dataloader_num_workers=2,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",              # 8-bit optimizer to save VRAM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print(f'Training config: {training_args.num_train_epochs} epochs, '
      f'batch={training_args.per_device_train_batch_size}x{training_args.gradient_accumulation_steps} (eff={training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}), '
      f'lr={training_args.learning_rate}, max_seq={MAX_SEQ_LEN}')

## 8. Train

In [ ]:
trainer.train()

## 9. Evaluate on Test Set

For a generative model, we evaluate by:
1. Generating narratives for test inputs
2. Comparing against ground truth with ROUGE scores
3. Checking quality rules (contains score, specific signals, 2-3 sentences)

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer

# Generate narratives for test set
model.eval()
generated = []
references = []

print('Generating narratives for test set...')
for i, rec in enumerate(test_raw):
    user_text = strip_snapshot_tier(rec['input'])
    ref_narrative = rec['label']['narrative']

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the generated part
    gen_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    generated.append(gen_text)
    references.append(ref_narrative)

    if i < 3:
        print(f'\n--- Test #{i+1} ---')
        print(f'Generated: {gen_text[:200]}')
        print(f'Reference: {ref_narrative[:200]}')

    if (i + 1) % 50 == 0:
        print(f'  Generated {i+1}/{len(test_raw)}')

print(f'\nGenerated {len(generated)} narratives')

## 9b. ROUGE Scores & Quality Checks

In [ ]:
# ROUGE scores
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge1_scores, rouge2_scores, rougeL_scores = [], [], []

for gen, ref in zip(generated, references):
    scores = scorer.score(ref, gen)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

print('ROUGE Scores (mean):')
print(f'  ROUGE-1: {np.mean(rouge1_scores):.4f}')
print(f'  ROUGE-2: {np.mean(rouge2_scores):.4f}')
print(f'  ROUGE-L: {np.mean(rougeL_scores):.4f}')

# Quality checks (Section 5A.5 test labelling checks)
n_has_number = 0
n_correct_sentences = 0
n_has_score_ref = 0
n_no_bullets = 0

for i, (gen, rec) in enumerate(zip(generated, test_raw)):
    score = rec.get('scorer_score', 0.0)

    # Check 1: contains at least one number
    if re.search(r'\d+', gen):
        n_has_number += 1

    # Check 2: 2-3 sentences
    sents = sent_tokenize(gen)
    if 2 <= len(sents) <= 3:
        n_correct_sentences += 1

    # Check 3: references the risk score
    score_str = f'{score:.2f}'
    if score_str in gen or 'risk score' in gen.lower() or 'risk rating' in gen.lower():
        n_has_score_ref += 1

    # Check 4: no bullet points
    if '\n-' not in gen and '\n*' not in gen and '\n•' not in gen:
        n_no_bullets += 1

total = len(generated)
print(f'\nQuality Checks ({total} test samples):')
print(f'  Contains number:     {n_has_number}/{total} ({n_has_number/total*100:.1f}%)')
print(f'  2-3 sentences:       {n_correct_sentences}/{total} ({n_correct_sentences/total*100:.1f}%)')
print(f'  References score:    {n_has_score_ref}/{total} ({n_has_score_ref/total*100:.1f}%)')
print(f'  No bullet points:    {n_no_bullets}/{total} ({n_no_bullets/total*100:.1f}%)')

## 10. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: ROUGE score distributions
axes[0].boxplot([rouge1_scores, rouge2_scores, rougeL_scores],
                labels=['ROUGE-1', 'ROUGE-2', 'ROUGE-L'])
axes[0].set_title('ROUGE Score Distribution')
axes[0].set_ylabel('F-measure')
axes[0].grid(True, alpha=0.3)

# Plot 2: Generated narrative length distribution
gen_lens = [len(g.split()) for g in generated]
ref_lens = [len(r.split()) for r in references]
axes[1].hist(gen_lens, bins=20, alpha=0.6, label='Generated', color='#2196F3')
axes[1].hist(ref_lens, bins=20, alpha=0.6, label='Reference', color='#4CAF50')
axes[1].set_title('Narrative Length (words)')
axes[1].set_xlabel('Word count')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Sentence count distribution
gen_sent_counts = [len(sent_tokenize(g)) for g in generated]
axes[2].hist(gen_sent_counts, bins=range(1, 8), alpha=0.7, color='#FF9800', edgecolor='black')
axes[2].axvline(x=2, color='green', linestyle='--', label='Target min (2)')
axes[2].axvline(x=3, color='green', linestyle='--', label='Target max (3)')
axes[2].set_title('Sentence Count Distribution')
axes[2].set_xlabel('Number of sentences')
axes[2].set_ylabel('Count')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_PLOTS}/reasoner_eval_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plots saved to {DRIVE_PLOTS}/')

In [ ]:
# Training loss curve
log_history = trainer.state.log_history
train_losses = [(h['step'], h['loss']) for h in log_history if 'loss' in h and 'eval_loss' not in h]
eval_losses = [(h['step'], h['eval_loss']) for h in log_history if 'eval_loss' in h]

fig, ax = plt.subplots(figsize=(10, 5))
if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label='Train Loss', alpha=0.7, color='#2196F3')
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, 'o-', label='Val Loss', color='#F44336', markersize=8)

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Reasoner Training — Loss Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_PLOTS}/reasoner_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Save LoRA Adapter & Merge (Optional)

Save the LoRA adapter to Google Drive for persistence.
Optionally merge into a full model for easier deployment.

In [ ]:
# Save LoRA adapter to Drive
ADAPTER_DIR = f'{DRIVE_DIR}/lora_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'LoRA adapter saved to {ADAPTER_DIR}')

# List saved files
for f in os.listdir(ADAPTER_DIR):
    size_mb = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1024 / 1024
    print(f'  {f}: {size_mb:.1f} MB')

In [ ]:
# Optional: Merge LoRA into base model for single-file deployment
# This creates a ~15GB merged model — only do this if you need it

MERGE = True  # Set to False to skip merging

if MERGE:
    MERGED_DIR = f'{DRIVE_DIR}/merged_model'
    print('Merging LoRA adapter into base model...')
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    print(f'Merged model saved to {MERGED_DIR}')

    for f in os.listdir(MERGED_DIR):
        size_mb = os.path.getsize(os.path.join(MERGED_DIR, f)) / 1024 / 1024
        print(f'  {f}: {size_mb:.1f} MB')
else:
    print('Skipping merge — using LoRA adapter only')

## 12. Sample Generated Narratives (Side-by-Side)

In [ ]:
# Show 10 random test examples: generated vs reference
import random as _rng
_rng.seed(42)
sample_indices = _rng.sample(range(len(generated)), min(10, len(generated)))

for idx in sample_indices:
    rec = test_raw[idx]
    print(f'\n{"="*80}')
    print(f'Issue #{rec["issue_number"]} ({rec["snapshot_tier"]}) | '
          f'{rec.get("scorer_band", "?")} | score={rec.get("scorer_score", 0):.2f}')
    print(f'{"="*80}')
    print(f'GENERATED:\n{generated[idx]}')
    print(f'\nREFERENCE:\n{references[idx]}')
    rouge = scorer.score(references[idx], generated[idx])
    print(f'\nROUGE-L: {rouge["rougeL"].fmeasure:.3f}')

## 13. Summary

| Parameter | Value |
|-----------|-------|
| Base model | Qwen2.5-7B-Instruct |
| Task | Causal LM / SFT |
| LoRA r | 32 |
| LoRA alpha | 64 |
| Target modules | q_proj, v_proj, k_proj, o_proj, gate_proj, up_proj |
| Epochs | 3 |
| Batch size | 8 |
| Learning rate | 1e-4 |
| Max seq length | 2048 |
| Train samples | ~2,044 |
| Val samples | ~275 |
| Test samples | ~259 |